In [3]:
#!/usr/bin/env python3
import os
import glob
import re
import random
import logging
from dataclasses import dataclass, replace
from typing import List, Tuple, Optional, Dict, Any
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform as rt_transform
from math import radians, sin, cos, sqrt, atan2
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter
from scipy.spatial import Delaunay
from scipy.interpolate import griddata

# Plotting defaults
sns.set_context("paper", font_scale=1.4)
sns.set_style("whitegrid", {'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.5})
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']

COLORS = sns.color_palette("deep")
OBS_COLOR = "#333333"
PRED_COLOR = COLORS[0]
FILL_COLOR = COLORS[0]

# Simple mesh cache to avoid re-triangulating same template repeatedly
_mesh_cache: Dict[str, Tuple[pd.DataFrame, dict]] = {}

####################################
@dataclass
class Config:
    # INPUT PATHS
    stations_dir: str = r"C:\Downloads\UHI_research\original_2023\filtered_2023_aug_hourly.csv"
    static_tif_folder: str = r"C:\Downloads\UHI_research\Static_features"

    # OUTPUT PATH
    base_output_dir: str = r"C:\Downloads\UHI_research\Output_LOOCV_Irregular_MeshTa"

    # SETTINGS
    years: Tuple[int, ...] = (2023,)
    month_start: int = 8
    month_end: int = 8
    month_day_start: int = 1        
    month_day_end: int = 30          
    min_sensor_count: int = 8
    apply_time_filter_to_test: bool = True

    # Graph settings
    k_neighbors_sensors: int = 6
    max_sensor_edge_km: Optional[float] = None
    make_sensor_graph_undirected: bool = True
    k_neighbors_holdout: int = 6
    max_holdout_edge_km: Optional[float] = None
    holdout_edges_undirected: bool = True

    # Model
    hidden_dim: int = 32
    heads: int = 3
    dropout: float = 0.4

    # Training
    learning_rate: float = 0.003
    epochs: int = 200
    batch_size: int = 32
    weight_decay: float = 1e-3
    seed: int = 42
    laplacian_lambda: float = 2e-3

    # Output settings
    smooth_sigma_px: float = 3.0
    smooth_passes: int = 2
    n_export_samples: int = 10
    plot_dpi: int = 450

    # CRS
    sensor_crs: str = "EPSG:4326"

    # IRREGULAR MESH SETTINGS
    # Replaces grid_downsample
    mesh_num_nodes: int = 10000  # Total points in the irregular mesh
    mesh_center_bias: float = 1.5 # Higher value = more points in center, fewer at edges
    template_tif_for_grid: Optional[str] = None

    # Filters
    static_name_filters: Tuple[str, ...] = (
        "LST_Terra",
        "LST_Aqua",
        "Montreal_Albedo",
        "NDVI_Montreal",
        "DEM",
    )

    # Early stopping
    es_patience: int = 40
    es_min_delta: float = 0.0

    # Range (plotting)
    temp_min: float = 10.0
    temp_max: float = 45.0

    min_train_sensors_at_ts_for_test: int = 8
    export_recon_tifs: bool = True
    output_dir: str = ""

    # PHYSICAL LIMITS 
    temp_physical_min: float = -20.0
    temp_physical_max: float = 55.0

    # RESAMPLING
    resample_freq: str = "H"
    resample_agg: str = "mean"
    
    # FUNCTIONAL GRAPH SETTINGS
    # 0.0 = Pure Physical Distance (Current)
    # 1.0 = Pure Functional Similarity (Ignores location)
    # 0.5 = Hybrid
    functional_weight: float = 0.8  
      
    # List of static features to use for similarity (must match TIF names)
    similarity_features: Tuple[str, ...] = ("NDVI_Montreal", "Montreal_Albedo", "DEM")
    # DAILY fallback strategy for rasters
    daily_fallback: str = "nearest"  # or "none"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
###################################
def safe_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        return pd.DataFrame()
    for enc in ["utf-8", "ISO-8859-1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="cp1252")


def find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    return None


def filter_tifs_by_name(tif_files: List[str], filters: Tuple[str, ...]) -> List[str]:
    if filters:
        tif_files = [p for p in tif_files if any(f.lower() in os.path.basename(p).lower() for f in filters)]
    return tif_files

def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

def reproject_points(lons: np.ndarray, lats: np.ndarray, src_crs: str, dst_crs: str):
    xs, ys = rt_transform(src_crs, dst_crs, lons.tolist(), lats.tolist())
    return np.array(xs), np.array(ys)

def smooth_image(img: np.ndarray, sigma: float, passes: int) -> np.ndarray:
    out = img.copy()
    for _ in range(max(1, int(passes))):
        out = gaussian_filter(out, sigma=float(sigma))
    return out

###################################
def safe_nanmedian(arr: np.ndarray, fallback: float = 0.0) -> float:
    if arr.size == 0 or np.all(np.isnan(arr)):
        return fallback
    med = np.nanmedian(arr)
    return fallback if np.isnan(med) else float(med)
###################################
class FeatureManager:
    def __init__(self):
        self.all_raw_cols = []
        self.features = {}
        self.sorted_base_names = []

    def parse_and_register(self, file_path: str):
        base = os.path.splitext(os.path.basename(file_path))[0]
        self.all_raw_cols.append(base)
        m_month_num = re.search(r"_([0-9]{1,2})$", base)
        m_month_txt = re.search(r"_(jan|feb|mar|apr|may|jun|jul|aug|sep|sept|september|oct|nov|dec)", base, re.IGNORECASE)
        m_date_iso = re.search(r"_(\d{4}-\d{2}-\d{2})$", base)
        m_date_compact = re.search(r"_(\d{8})$", base)
        months = {
            "jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
            "jul": 7, "aug": 8, "sep": 9, "sept": 9, "september": 9,
            "oct": 10, "nov": 11, "dec": 12
        }

        def reg_daily(clean, dt):
            self.features.setdefault(clean, {"type": "daily", "cols": {}})["cols"][dt.date()] = base

        if m_date_iso or m_date_compact:
            dstr = (m_date_iso or m_date_compact).group(1)
            dt = pd.to_datetime(dstr, format="%Y-%m-%d", errors="coerce")
            if pd.isna(dt):
                dt = pd.to_datetime(dstr, format="%Y%m%d", errors="coerce")
            clean = base[:(m_date_iso or m_date_compact).start()]
            reg_daily(clean, dt)
            return

        if m_month_num:
            m = int(m_month_num.group(1))
            if 1 <= m <= 12:
                clean = base[:m_month_num.start()]
                self.features.setdefault(clean, {"type": "monthly", "cols": {}})["cols"][m] = base
                return

        if m_month_txt:
            sub = m_month_txt.group(1).lower()
            if sub in months:
                clean = base[:m_month_txt.start()]
                self.features.setdefault(clean, {"type": "monthly", "cols": {}})["cols"][months[sub]] = base
                return

        self.features[base] = {"type": "static", "col": base}

    def finalize(self):
        self.sorted_base_names = sorted(self.features.keys())

    def get_cols_for_datetime(self, ts: pd.Timestamp, daily_fallback: str = "nearest"):
        out = []
        for base in self.sorted_base_names:
            f = self.features[base]
            if f["type"] == "static":
                out.append(f["col"])
            elif f["type"] == "monthly":
                if ts.month not in f["cols"]:
                    raise ValueError(f"Missing monthly feature '{base}' for month {ts.month}")
                out.append(f["cols"][ts.month])
            elif f["type"] == "daily":
                cols = f["cols"]
                d = ts.date()
                if d in cols:
                    out.append(cols[d])
                elif daily_fallback == "nearest":
                    nearest = min(cols.keys(), key=lambda dd: abs(pd.Timestamp(dd) - ts.normalize()))
                    out.append(cols[nearest])
                else:
                    raise ValueError(f"Missing daily feature '{base}' for date {d}")
        return out

    def get_input_dim(self):
        return len(self.sorted_base_names)
######################################
def _select_similarity_feature_cols(df_cols, fm: "FeatureManager", cfg: "Config"):
    """
    Intersect available columns with cfg.similarity_features.
    """
    requested = set(cfg.similarity_features)
    present = set(df_cols)
    use_cols = sorted(list(requested & present))
    return use_cols

def _knn_hybrid_sensor_to_grid_edges(snap_df, grid_df, fm: "FeatureManager", cfg: "Config", k_eff: int, cols_for_ts: list):
    """
    Build hybrid sensor->mesh kNN edges.
    """
    use_cols = _select_similarity_feature_cols(cols_for_ts, fm, cfg)

    S_coords = snap_df[["latitude", "longitude"]].to_numpy(dtype=float)
    G_coords = grid_df[["latitude", "longitude"]].to_numpy(dtype=float)

    if use_cols:
        S_feats = snap_df[use_cols].to_numpy(dtype=float)
        G_feats = grid_df[use_cols].to_numpy(dtype=float)
    else:
        S_feats = np.zeros((len(S_coords), 0), dtype=float)
        G_feats = np.zeros((len(G_coords), 0), dtype=float)

    coord_scale = 100.0 * (1.0 - float(cfg.functional_weight))
    feat_scale  = 5.0   * float(cfg.functional_weight)

    S_hybrid = np.hstack([S_coords * coord_scale, S_feats * feat_scale])
    G_hybrid = np.hstack([G_coords * coord_scale, G_feats * feat_scale])

    if len(S_hybrid) == 0 or len(G_hybrid) == 0:
        return torch.empty((2,0), dtype=torch.long), torch.empty((2,0), dtype=torch.long)

    k_eff = max(1, min(k_eff, len(S_hybrid)))
    nn = NearestNeighbors(n_neighbors=k_eff).fit(S_hybrid)
    _, idxs = nn.kneighbors(G_hybrid)

    sources = idxs.flatten()                      # sensor local indices [0..Ns-1]
    targets = np.repeat(np.arange(len(G_hybrid)), k_eff)  # grid local indices [0..Ng-1]

    e_s2g_local = torch.tensor([sources, targets], dtype=torch.long)
    e_g2s_local = torch.tensor([targets, sources], dtype=torch.long)
    return e_s2g_local, e_g2s_local

def build_sensor_grid_mesh_graph_for_timestamp(
    ts: "pd.Timestamp",
    cfg: "Config",
    fm: "FeatureManager",
    train_norm: "pd.DataFrame",
    unique_train_stations: "pd.DataFrame",
    static_stats: dict,
    coord_scaler,
    template_tif: str,
    raster_index: dict,
    k_sensor_to_grid: int = None
) -> Data:
    """
    Constructs a heterogeneous graph at a single timestamp ts using Irregular Mesh.
    """
    if k_sensor_to_grid is None:
        k_sensor_to_grid = cfg.k_neighbors_sensors

    # Sensors present at ts
    snap = train_norm[train_norm["dt"] == ts].copy()
    if snap.empty:
        raise ValueError(f"No sensor readings at {ts}")

    present_ids = snap["id_station"].unique().tolist()
    sensors_df = unique_train_stations[unique_train_stations["id_station"].isin(present_ids)].copy().reset_index(drop=True)
    if sensors_df.empty:
        raise ValueError("No matching stations found in unique_train_stations for current timestamp.")

    # Build sensor-sensor edges
    ei_ss, ew_ss = build_functional_sensor_graph(
        stations_df=sensors_df,
        k=cfg.k_neighbors_sensors,
        max_dist=cfg.max_sensor_edge_km,
        config=cfg,
        feature_cols=fm.all_raw_cols
    )

    # BUILD IRREGULAR MESH ---
    grid_df, grid_meta = build_irregular_mesh(cfg, template_tif, num_nodes=cfg.mesh_num_nodes)
    
    # Sample/normalize static rasters for mesh points
    grid_df = sample_static_tifs_for_grid(grid_df, cfg, fm, static_stats, coord_scaler, ts=ts, raster_index=raster_index)

    # Mesh adjacency (Delaunay)
    e_grid = build_mesh_edges_delaunay(grid_meta)

    # Features: static, time, coords, temp_in for sensors; grid temp_in=0
    cols_for_ts = fm.get_cols_for_datetime(ts, daily_fallback=cfg.daily_fallback)

    # Sensors: static features
    X_static_s = sensors_df[cols_for_ts].to_numpy(dtype=np.float32)
    # Time
    tfeat = np.array([
        np.sin(2*np.pi*ts.hour/24), np.cos(2*np.pi*ts.hour/24),
        np.sin(4*np.pi*ts.hour/24), np.cos(4*np.pi*ts.hour/24),
        np.sin(2*np.pi*ts.dayofyear/365), np.cos(2*np.pi*ts.dayofyear/365)
    ], dtype=np.float32)
    X_time_s = np.tile(tfeat, (len(sensors_df), 1))
    # Coords
    X_coord_s = sensors_df[["lat_norm", "lon_norm"]].to_numpy(dtype=np.float32)
    # Temp input (standardized)
    y_mean = train_norm["Ta"].mean()
    y_std = max(train_norm["temperature"].std(), 1e-3)
    temp_map = dict(zip(snap["id_station"], ((snap["temperature"] - y_mean) / y_std).to_numpy()))
    temp_s = np.array([temp_map[i] for i in sensors_df["id_station"].tolist()], dtype=np.float32).reshape(-1, 1)

    # Grid: static, time, coords, temp_in=0
    X_static_g = grid_df[cols_for_ts].to_numpy(dtype=np.float32)
    X_time_g = np.tile(tfeat, (len(grid_df), 1))
    X_coord_g = grid_df[["lat_norm", "lon_norm"]].to_numpy(dtype=np.float32)
    temp_g = np.zeros((len(grid_df), 1), dtype=np.float32)

    X_s = np.concatenate([X_static_s, X_time_s, X_coord_s, temp_s], axis=1)
    X_g = np.concatenate([X_static_g, X_time_g, X_coord_g, temp_g], axis=1)
    X_all = np.concatenate([X_s, X_g], axis=0)

    Ns = len(sensors_df)
    Ng = len(grid_df)

    # Sensor<->Mesh edges (hybrid distance)
    e_s2g_local, e_g2s_local = _knn_hybrid_sensor_to_grid_edges(
        snap_df=snap.merge(sensors_df[["id_station"]], on="id_station", how="inner"),
        grid_df=grid_df,
        fm=fm,
        cfg=cfg,
        k_eff=min(k_sensor_to_grid, Ns) if Ns > 0 else 1,
        cols_for_ts=cols_for_ts
    )
    # Shift grid indices by +Ns
    e_s2g = torch.stack([e_s2g_local[0], e_s2g_local[1] + Ns], dim=0)
    e_g2s = torch.stack([e_g2s_local[0] + Ns, e_g2s_local[1]], dim=0)

    # Grid lattice shifted
    e_grid_global = e_grid.clone() + Ns

    # Final edge_index
    edge_index = torch.cat([ei_ss, e_s2g, e_g2s, e_grid_global], dim=1)

    n_ss = ei_ss.shape[1]
    n_s2g = e_s2g.shape[1]
    n_g2s = e_g2s.shape[1]
    n_grid = e_grid_global.shape[1]
    ew_rest = torch.ones(n_s2g + n_g2s + n_grid, dtype=torch.float32)
    edge_weight = torch.cat([ew_ss, ew_rest], dim=0)

    y = np.full((Ns + Ng,), np.nan, dtype=np.float32)
    y[:Ns] = temp_s.flatten()
    y = torch.from_numpy(y)

    sensor_mask = torch.zeros(Ns + Ng, dtype=torch.bool); sensor_mask[:Ns] = True
    grid_mask = ~sensor_mask
    loss_mask = torch.zeros((Ns + Ng, 1), dtype=torch.float32); loss_mask[:Ns] = 1.0

    data = Data(
        x=torch.from_numpy(X_all).float(),
        edge_index=edge_index.long(),
        edge_weight=edge_weight.float(),
        y=y.float()
    )
    data.Ns = Ns; data.Ng = Ng
    data.sensor_mask = sensor_mask
    data.grid_mask = grid_mask
    data.loss_mask = loss_mask
    data.sensor_ids = sensors_df["id_station"].tolist()
    data.grid_meta = grid_meta
    data.grid_latlon = grid_df[["latitude", "longitude"]].to_numpy()

    return data

def load_all_csvs(config: Config) -> pd.DataFrame:
    files = [config.stations_dir] if os.path.isfile(config.stations_dir) else [
        f for f in glob.glob(os.path.join(config.stations_dir, "*.csv"))
        if "rest_excluding" not in os.path.basename(f).lower()
    ]
    if not files:
        raise ValueError(f"No valid CSVs found in {config.stations_dir}")

    dfs = []
    for f in files:
        df = safe_read_csv(f)
        df.columns = df.columns.str.strip()

        lat_col = find_col(df, ["latitude", "lat", "y", "sensor_lat"])
        lon_col = find_col(df, ["longitude", "lon", "long", "x", "sensor_lon"])
        id_col  = find_col(df, ["id_station", "station", "station_id", "id", "sensor_id", "name", "station_name"])
        date_col= find_col(df, ["date", "datetime", "timestamp", "time", "dt"])
        temp_col= find_col(df, ["temperature"])
        if not all([lat_col, lon_col, id_col, date_col, temp_col]):
            continue

        df = df.rename(columns={
            lat_col: "latitude", lon_col: "longitude", id_col: "id_station", date_col: "date", temp_col: "temperature"
        })
        if "Unnamed: 0" in df.columns:
            df = df.drop(columns=["Unnamed: 0"])

        df["dt"] = pd.to_datetime(df["date"], errors="coerce")
        df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
        df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")
        df["id_station"] = df["id_station"].astype(str)

        mask = (
            df["dt"].dt.year.isin(config.years) &
            (df["dt"].dt.month >= config.month_start) &
            (df["dt"].dt.month <= config.month_end) &
            (df["dt"].dt.day >= config.month_day_start) &
            (df["dt"].dt.day <= config.month_day_end)
        )
        df = df.loc[mask].dropna(subset=["temperature", "latitude", "longitude"])

        if not df.empty:
            dfs.append(df)

    if not dfs:
        raise ValueError("No data remained after filtering.")

    big_df = pd.concat(dfs, ignore_index=True)

    h = big_df["dt"].dt.hour.to_numpy()
    doy = big_df["dt"].dt.dayofyear.to_numpy()
    big_df["hour_sin"]  = np.sin(2*np.pi*h/24)
    big_df["hour_cos"]  = np.cos(2*np.pi*h/24)
    big_df["hour_sin2"] = np.sin(4*np.pi*h/24)
    big_df["hour_cos2"] = np.cos(4*np.pi*h/24)
    big_df["doy_sin"]   = np.sin(2*np.pi*doy/365)
    big_df["doy_cos"]   = np.cos(2*np.pi*doy/365)

    big_df = big_df[
        (big_df["temperature"] >= config.temp_physical_min) &
        (big_df["temperature"] <= config.temp_physical_max)
    ].copy()

    if config.min_sensor_count > 0:
        counts = big_df.groupby("dt")["id_station"].nunique()
        valid_ts = counts[counts >= config.min_sensor_count].index
        big_df = big_df[big_df["dt"].isin(valid_ts)].copy()
        if big_df.empty:
            raise ValueError(f"No timestamps found with >= {config.min_sensor_count} sensors!")

    return big_df


def sample_static_and_init_features(big_df: pd.DataFrame, config: Config) -> Tuple[pd.DataFrame, FeatureManager]:
    unique_stations = big_df[["id_station", "latitude", "longitude"]].drop_duplicates().reset_index(drop=True)
    tif_files = filter_tifs_by_name(glob.glob(os.path.join(config.static_tif_folder, "*.tif")), config.static_name_filters)
    if not tif_files:
        raise FileNotFoundError("No TIF files found.")

    fm = FeatureManager()
    for t in tif_files:
        fm.parse_and_register(t)
    fm.finalize()

    sampled_cols: Dict[str, np.ndarray] = {}
    for tif_path in tif_files:
        base = os.path.splitext(os.path.basename(tif_path))[0]
        with rasterio.open(tif_path) as src:
            if src.crs.to_string() != config.sensor_crs:
                xs, ys = reproject_points(
                    unique_stations["longitude"].to_numpy(),
                    unique_stations["latitude"].to_numpy(),
                    config.sensor_crs,
                    src.crs.to_string()
                )
                coords = list(zip(xs, ys))
            else:
                coords = list(zip(unique_stations["longitude"], unique_stations["latitude"]))

            vals = np.array([v[0] for v in src.sample(coords)], dtype="float32")
            if src.nodata is not None:
                vals = np.where(vals == src.nodata, np.nan, vals)

            if np.all(np.isnan(vals)):
                arr = src.read(1, masked=True).astype("float32")
                arr_data = np.ma.filled(arr, np.nan)
                global_med = safe_nanmedian(arr_data, fallback=0.0)
                vals = np.full_like(vals, global_med, dtype="float32")

            sampled_cols[base] = vals

    unique_stations = pd.concat([unique_stations, pd.DataFrame(sampled_cols)], axis=1)

    for c in fm.all_raw_cols:
        med = safe_nanmedian(unique_stations[c].to_numpy(), fallback=0.0)
        unique_stations[c] = np.where(np.isnan(unique_stations[c]), med, unique_stations[c])

    final_df = big_df.merge(unique_stations[["id_station"] + fm.all_raw_cols], on="id_station", how="inner")
    return final_df, fm
########################################
def fit_normalizers_fast(train_df, all_raw_cols):
    stats = {c: (train_df[c].mean(), train_df[c].std() if train_df[c].std() > 1e-9 else 1.0) for c in all_raw_cols}
    scaler = MinMaxScaler().fit(train_df[["latitude", "longitude"]])
    return stats, scaler


def normalize_data(df, all_raw_cols, stats, scaler):
    out = df.copy()
    for c in all_raw_cols:
        m, s = stats[c]
        out[c] = (out[c] - m) / s
    coords_scaled = scaler.transform(out[["latitude", "longitude"]])
    out = out.assign(lat_norm=coords_scaled[:, 0], lon_norm=coords_scaled[:, 1])
    return out


def build_functional_sensor_graph(stations_df, k, max_dist, config: Config, feature_cols: List[str]):
    coords = stations_df[["latitude", "longitude"]].to_numpy()

    requested = set(feature_cols)
    allowed = set(config.similarity_features)
    present = set(stations_df.columns)
    use_feats = sorted(list((requested & allowed) & present))

    if not use_feats:
        feature_matrix = np.zeros((len(coords), 0), dtype=float)
    else:
        feature_matrix = stations_df[use_feats].to_numpy(dtype=float)

    coord_scale_factor = 100.0 * (1.0 - float(config.functional_weight))
    feat_scale_factor = 5.0 * float(config.functional_weight)

    hybrid_vectors = np.hstack([
        coords * coord_scale_factor,
        feature_matrix * feat_scale_factor
    ])
    n = len(hybrid_vectors)
    if n < 2:
        return torch.empty((2, 0), dtype=torch.long), torch.empty(0)

    k_eff = min(k + 1, n)
    nbrs = NearestNeighbors(n_neighbors=k_eff).fit(hybrid_vectors)
    _, idx = nbrs.kneighbors(hybrid_vectors)

    src, dst, wts = [], [], []
    for i in range(n):
        for j in idx[i, 1:]:
            d_phys = haversine_km(coords[i, 0], coords[i, 1], coords[j, 0], coords[j, 1])
            if max_dist and d_phys > max_dist:
                continue
            if feature_matrix.shape[1] > 0:
                feat_dist = np.linalg.norm(feature_matrix[i] - feature_matrix[j])
            else:
                feat_dist = 0.0
            w = 1.0 / (d_phys + 0.1 * feat_dist + 1e-3)
            src.append(i); dst.append(int(j)); wts.append(w)
            src.append(int(j)); dst.append(i); wts.append(w)

    return torch.tensor([src, dst], dtype=torch.long), torch.tensor(wts, dtype=torch.float32)
################################
def snapshot_to_data(df_ts, stations_train, id_to_idx, fm: FeatureManager, edge_index, edge_weight, y_mean, y_std, config):
    N = len(stations_train)
    r0 = df_ts.iloc[0]
    try:
        cols_to_use = fm.get_cols_for_datetime(r0["dt"], daily_fallback=config.daily_fallback)
    except ValueError:
        return None
    X_static = stations_train[cols_to_use].values.astype(np.float32)
    X_time = np.tile([r0.hour_sin, r0.hour_cos, r0.hour_sin2, r0.hour_cos2, r0.doy_sin, r0.doy_cos], (N, 1)).astype(np.float32)
    X_coord = stations_train[["lat_norm", "lon_norm"]].values.astype(np.float32)
    temp_in = np.zeros(N, dtype=np.float32)
    y = np.zeros(N, dtype=np.float32)
    availability_mask = np.zeros(N, dtype=np.float32)
    loss_mask = np.zeros(N, dtype=np.float32)
    available = df_ts[df_ts["id_station"].isin(id_to_idx.keys())]
    if available.empty or available["id_station"].nunique() < 2:
        return None
    indices = [id_to_idx[sid] for sid in available["id_station"]]
    vals = (available["temperature"].values - y_mean) / y_std
    temp_in[indices] = vals; y[indices] = vals; availability_mask[indices] = 1.0
    present_indices = np.where(availability_mask == 1)[0]
    if len(present_indices) < 2:
        return None
    mask_count = max(1, min(len(present_indices) - 1, int(0.2 * len(present_indices))))
    masked_idx = np.random.choice(present_indices, size=mask_count, replace=False)
    temp_in[masked_idx] = 0.0
    loss_mask[masked_idx] = 1.0
    X = np.concatenate([X_static, X_time, X_coord, temp_in[:, None]], axis=1)
    data = Data(x=torch.from_numpy(X), edge_index=edge_index, y=torch.from_numpy(y), edge_weight=edge_weight)
    data.loss_mask = torch.from_numpy(loss_mask[:, None])
    return data


def make_dataloader(train_df, stations_train, id_to_idx, fm: FeatureManager, edge_index, edge_weight, config, y_mean, y_std):
    snaps = []
    for _, group in train_df.groupby("dt"):
        if len(group) < 2:
            continue
        d = snapshot_to_data(group, stations_train, id_to_idx, fm, edge_index, edge_weight, y_mean, y_std, config)
        if d:
            snaps.append(d)
    if not snaps:
        raise ValueError("No valid snapshots")
    return DataLoader(snaps, batch_size=config.batch_size, shuffle=True), len(snaps)
###################################
class DualStreamGAT(torch.nn.Module):
    def __init__(self, num_static: int, num_time: int, hidden_dim: int, heads: int = 3, dropout: float = 0.4):
        super().__init__()
        self.geo_gat = GATv2Conv(in_channels=2, out_channels=hidden_dim, heads=heads, dropout=dropout, concat=True)
        self.sem_in_dim = num_static + num_time + 1
        self.sem_gat = GATv2Conv(in_channels=self.sem_in_dim, out_channels=hidden_dim, heads=heads, dropout=dropout, concat=True)
        fusion_dim = (hidden_dim * heads) * 2
        self.norm = torch.nn.LayerNorm(fusion_dim)
        self.lin_out = torch.nn.Linear(fusion_dim, 1)

    def forward(self, x_semantic, x_coords, edge_index):
        h_geo = self.geo_gat(x_coords, edge_index)
        h_sem = self.sem_gat(x_semantic, edge_index)
        combined = torch.cat([h_geo, h_sem], dim=1)
        combined = self.norm(combined)
        combined = F.elu(combined)
        return self.lin_out(combined)
##########################################
def train_model(train_df, stations_train, id_to_idx, fm: FeatureManager, edge_index, edge_weight, config):
    y_mean = train_df["temperature"].mean()
    y_std = max(train_df["temperature"].std(), 1e-3)
    loader, n_snaps = make_dataloader(train_df, stations_train, id_to_idx, fm, edge_index, edge_weight, config, y_mean, y_std)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_static = fm.get_input_dim(); n_time = 6
    model = DualStreamGAT(n_static, n_time, config.hidden_dim, config.heads, config.dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    best_loss = float('inf'); patience_ctr = 0; history = {"train_loss": []}
    for epoch in range(config.epochs):
        model.train(); total_loss = 0
        for batch in loader:
            batch = batch.to(device); opt.zero_grad()
            x_static_time = batch.x[:, :n_static + n_time]
            x_temp_in = batch.x[:, -1:]
            x_sem = torch.cat([x_static_time, x_temp_in], dim=1)
            x_geo = batch.x[:, n_static + n_time : n_static + n_time + 2]
            out = model(x_sem, x_geo, batch.edge_index)
            err = (out.squeeze() - batch.y) ** 2
            mse = (err * batch.loss_mask.squeeze()).sum() / (batch.loss_mask.sum() + 1e-6)
            src, dst = batch.edge_index
            diff = (out[src] - out[dst])**2
            lap = (batch.edge_weight[:, None].to(device) * diff).mean()
            loss = mse + config.laplacian_lambda * lap
            loss.backward(); opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader); history["train_loss"].append(avg_loss)
        if avg_loss < best_loss - config.es_min_delta:
            best_loss = avg_loss; patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= config.es_patience:
                break
    return model, history, y_mean, y_std, device
#######################################
def save_trained_model(output_dir: str, model: torch.nn.Module, y_mean: float, y_std: float, fm: FeatureManager, config: Config):
    os.makedirs(output_dir, exist_ok=True)
    save_path = os.path.join(output_dir, "model_state.pth")
    torch.save({
        "state_dict": model.state_dict(),
        "y_mean": float(y_mean),
        "y_std": float(y_std),
        "feature_bases": fm.sorted_base_names,
        "n_static": fm.get_input_dim(),
        "n_time": 6,
        "hidden_dim": config.hidden_dim,
        "heads": config.heads,
        "dropout": config.dropout,
        "model_class": "DualStreamGAT"
    }, save_path)
    print(f"Saved model to: {save_path}")
#################################
@torch.no_grad()
def evaluate_holdout(model, train_df, hold_df, stations_train, id_to_idx, fm: FeatureManager, ei_train, ew_train, y_mean, y_std, config, device, hold_name):
    model.eval()
    common_ts = sorted(list(set(train_df["dt"]) & set(hold_df["dt"])))
    if not common_ts:
        return None, None

    y_true, y_pred, dates = [], [], []
    hold_row = hold_df.iloc[0]

    full_ei = torch.tensor([ei_train[0].tolist(), ei_train[1].tolist()], dtype=torch.long).to(device)

    N_train = len(stations_train)
    n_static = fm.get_input_dim()
    n_time = 6

    k_h = max(1, min(config.k_neighbors_holdout, N_train))
    coords_train = stations_train[["latitude", "longitude"]].to_numpy()
    nn = NearestNeighbors(n_neighbors=k_h).fit(coords_train)

    hold_lat = float(hold_row["latitude"])
    hold_lon = float(hold_row["longitude"])
    _, idxs = nn.kneighbors(np.array([[hold_lat, hold_lon]]), return_distance=True)

    if config.max_holdout_edge_km is not None:
        neighbors = []
        for j in idxs[0]:
            d_phys = haversine_km(hold_lat, hold_lon, coords_train[j, 0], coords_train[j, 1])
            if d_phys <= config.max_holdout_edge_km:
                neighbors.append(int(j))
        if not neighbors and len(idxs[0]) > 0:
            neighbors = [int(idxs[0][0])]
    else:
        neighbors = [int(j) for j in idxs[0]]

    if neighbors:
        hold_node_idx = N_train
        hold_src = neighbors
        hold_dst = [hold_node_idx] * len(neighbors)
        if config.holdout_edges_undirected:
            ei_hold = torch.tensor([hold_src + hold_dst, hold_dst + hold_src], dtype=torch.long).to(device)
        else:
            ei_hold = torch.tensor([hold_src, hold_dst], dtype=torch.long).to(device)
        full_ei = torch.cat([full_ei, ei_hold], dim=1)

    for ts in common_ts:
        if not (config.month_start <= ts.month <= config.month_end):
            continue

        snap_train = train_df[train_df["dt"] == ts]
        if len(snap_train) < config.min_train_sensors_at_ts_for_test:
            continue

        real_vals = hold_df[hold_df["dt"] == ts]["temperature"].values
        if real_vals.size == 0:
            continue
        real_val = real_vals[0]

        try:
            cols_to_use = fm.get_cols_for_datetime(ts, daily_fallback=config.daily_fallback)
        except ValueError:
            continue

        X_static_train = stations_train[cols_to_use].values
        X_static_hold = hold_row[cols_to_use].values.reshape(1, -1)
        X_static_all = np.vstack([X_static_train, X_static_hold]).astype(np.float32)

        X_coord_all = np.vstack([
            stations_train[["lat_norm", "lon_norm"]].values,
            hold_row[["lat_norm", "lon_norm"]].values.reshape(1, -1)
        ]).astype(np.float32)

        tfeat = np.array([
            np.sin(2*np.pi*ts.hour/24), np.cos(2*np.pi*ts.hour/24),
            np.sin(4*np.pi*ts.hour/24), np.cos(4*np.pi*ts.hour/24),
            np.sin(2*np.pi*ts.dayofyear/365), np.cos(2*np.pi*ts.dayofyear/365)
        ], dtype=np.float32)
        X_time = np.tile(tfeat, (N_train + 1, 1))

        temp_in = np.zeros(N_train + 1, dtype=np.float32)
        valid_rows = snap_train[snap_train["id_station"].isin(id_to_idx.keys())]
        if valid_rows.empty:
            continue
        idxs_train_nodes = [id_to_idx[i] for i in valid_rows["id_station"]]
        vals_std = (valid_rows["temperature"].values - y_mean) / y_std
        temp_in[idxs_train_nodes] = vals_std

        X = np.concatenate([X_static_all, X_time, X_coord_all, temp_in[:, None]], axis=1)
        x_tensor = torch.from_numpy(X).to(device)
        x_static_time = x_tensor[:, :n_static + n_time]
        x_temp_in = x_tensor[:, -1:]
        x_sem = torch.cat([x_static_time, x_temp_in], dim=1)
        x_geo = x_tensor[:, n_static + n_time : n_static + n_time + 2]

        out = model(x_sem, x_geo, full_ei).detach().cpu().numpy().flatten()
        pred = out[N_train] * y_std + y_mean

        y_true.append(real_val)
        y_pred.append(pred)
        dates.append(ts)

    if not y_true:
        return None, None

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    mae = np.mean(np.abs(y_pred - y_true))
    print(f"Holdout {hold_name}: RMSE={rmse:.3f}, MAE={mae:.3f}")

    cfg_out = config.output_dir or config.base_output_dir
    os.makedirs(cfg_out, exist_ok=True)

    plt.figure(figsize=(12, 4))
    plt.plot(dates, y_true, label="Observed", color=OBS_COLOR)
    plt.plot(dates, y_pred, label="Predicted", color=PRED_COLOR)
    plt.legend()
    plt.title(f"Holdout {hold_name}")
    plt.tight_layout()
    plt.savefig(os.path.join(cfg_out, "holdout_timeseries.png"), dpi=config.plot_dpi)
    plt.close()

    eval_df = pd.DataFrame({
        "dt": pd.to_datetime(dates),
        "station_id": str(hold_name),
        "observed": y_true,
        "predicted": y_pred,
    })
    eval_df["residual"] = eval_df["predicted"] - eval_df["observed"]
    eval_df["abs_error"] = eval_df["residual"].abs()
    eval_df = eval_df.sort_values("dt").reset_index(drop=True)
    csv_path = os.path.join(cfg_out, "holdout_observed_vs_predicted.csv")
    eval_df.to_csv(csv_path, index=False)
    print(f"Saved holdout observed vs predicted CSV to: {csv_path}")

    return rmse, mae
###################################
# IRREGULAR MESH FUNCTIONS
###################################
def build_irregular_mesh(config, template_tif, num_nodes=5000):
    """
    Generates an irregular mesh 
    (e.g., center of the map).

    This function caches the generated mesh per template_tif to avoid repeating the expensive Delaunay step.
    """
    global _mesh_cache
    if template_tif in _mesh_cache:
        # Return a copy of the stored dataframes/meta to avoid accidental in-place modifications
        cached_df, cached_meta = _mesh_cache[template_tif]
        return cached_df.copy(), cached_meta.copy()

    with rasterio.open(template_tif) as src:
        bounds = src.bounds
        min_x, min_y, max_x, max_y = bounds.left, bounds.bottom, bounds.right, bounds.top
        
        # 1. Define Density Function (Higher in center)
        def density_func(x, y):
            center_x, center_y = (min_x + max_x) / 2, (min_y + max_y) / 2
            max_dist = np.sqrt((max_x - min_x)**2 + (max_y - min_y)**2) / 2.0
            dist = np.sqrt((x - center_x)**2 + (y - center_y)**2)
            # Bias towards center
            prob = 1.0 - (dist / max_dist)
            # Apply power to sharpen the density gradient
            prob = np.power(np.clip(prob, 0.05, 1.0), config.mesh_center_bias)
            return prob

        points = []
        max_attempts = num_nodes * 10
        attempts = 0
        while len(points) < num_nodes and attempts < max_attempts:
            rx = np.random.uniform(min_x, max_x)
            ry = np.random.uniform(min_y, max_y)
            if np.random.rand() < density_func(rx, ry):
                points.append([rx, ry])
            attempts += 1
        
        points = np.array(points)
        if len(points) < 10:
             raise ValueError("Failed to generate enough mesh points. Check density function.")

        lons, lats = rt_transform(src.crs.to_string(), config.sensor_crs, points[:, 0], points[:, 1])

        df = pd.DataFrame({
            "longitude": lons, 
            "latitude": lats,
            "x_crs": points[:, 0],
            "y_crs": points[:, 1]
        })
        
        # Delaunay Triangulation
        tri = Delaunay(points)
        
        meta = {"tri": tri, "width": src.width, "height": src.height, "transform": src.transform, "crs": src.crs}
        # Cache
        _mesh_cache[template_tif] = (df.copy(), meta.copy())
        
        return df, meta

def build_mesh_edges_delaunay(grid_meta):
    """
    Builds graph edges from Delaunay triangulation simplices.
    Canonicalizes edges (min,max) to remove reversed duplicates, then
    returns bi-directional edges (both directions present).
    """
    tri = grid_meta["tri"]
    simplices = tri.simplices
    
    src = []
    dst = []
    
    for simplex in simplices:
        # Edges: A-B, B-C, C-A
        a, b, c = simplex[0], simplex[1], simplex[2]
        src.extend([a, b, c])
        dst.extend([b, c, a])
        
    edges = torch.tensor([src, dst], dtype=torch.long)
    # canonicalize columns so that (i,j) and (j,i) become identical (min,max)
    mins = torch.min(edges[0], edges[1])
    maxs = torch.max(edges[0], edges[1])
    canonical = torch.stack([mins, maxs], dim=0)
    unique = torch.unique(canonical, dim=1)
    # recreate both directions to have a directed edge list that includes both
    edges_bidir = torch.cat([unique, unique[[1,0], :]], dim=1)
    return edges_bidir

def sample_static_tifs_for_grid(grid, config, fm: FeatureManager, static_stats, coord_scaler, ts: pd.Timestamp, raster_index: Dict[str, str]):
    cols_for_ts = fm.get_cols_for_datetime(ts, daily_fallback=config.daily_fallback)
    g = grid.copy()
    for col_name in cols_for_ts:
        if col_name not in raster_index:
            raise FileNotFoundError(f"Raster not found for column {col_name} at {ts}")
        with rasterio.open(raster_index[col_name]) as src:
            if src.crs.to_string() != config.sensor_crs:
                # If grid has x_crs/y_crs and it matches raster CRS, use those directly to avoid reprojection error
                # For simplicity here, we reproject lat/lon unless CRS matches exactly
                xs, ys = reproject_points(g["longitude"].to_numpy(), g["latitude"].to_numpy(),
                                          config.sensor_crs, src.crs.to_string())
                coords = list(zip(xs, ys))
            else:
                coords = list(zip(g["longitude"].to_numpy(), g["latitude"].to_numpy()))
            
            vals = np.array([v[0] for v in src.sample(coords)], dtype="float32")
            if src.nodata is not None:
                vals = np.where(vals == src.nodata, np.nan, vals)

            if np.all(np.isnan(vals)):
                arr = src.read(1, masked=True).astype("float32")
                arr_data = np.ma.filled(arr, np.nan)
                global_med = safe_nanmedian(arr_data, fallback=0.0)
                vals = np.full_like(vals, global_med, dtype="float32")

            g[col_name] = vals

    for c in cols_for_ts:
        med = safe_nanmedian(g[c].to_numpy(), fallback=0.0)
        g[c] = np.where(np.isnan(g[c]), med, g[c])
        m, s = static_stats[c]
        g[c] = (g[c] - m) / s

    g[["lat_norm", "lon_norm"]] = coord_scaler.transform(g[["latitude", "longitude"]])
    return g

#########################################
def plot_mean_loss_across_folds(all_loss_histories, out_dir, plot_dpi=450):
    if not all_loss_histories:
        return
    max_len = max(len(h) for h in all_loss_histories)
    loss_matrix = np.full((len(all_loss_histories), max_len), np.nan, dtype=float)
    for i, h in enumerate(all_loss_histories):
        loss_matrix[i, :len(h)] = h
    mean_loss = np.nanmean(loss_matrix, axis=0)
    std_loss = np.nanstd(loss_matrix, axis=0)
    epochs = np.arange(1, max_len + 1)

    plt.figure(figsize=(10, 6))
    for h in all_loss_histories:
        plt.plot(range(1, len(h)+1), h, color='gray', alpha=0.15, linewidth=0.8)
    plt.plot(epochs, mean_loss, color='tab:blue', linewidth=2.5, label="Average Loss")
    plt.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, color='tab:blue', alpha=0.15, label="±1 Std Dev")
    plt.title("Average Training Loss Across LOOCV Folds", fontweight='bold')
    plt.xlabel("Epoch"); plt.ylabel("Loss (MSE + Laplacian)")
    plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, "mean_training_loss.svg"), format="svg")
    plt.savefig(os.path.join(out_dir, "mean_training_loss.png"), dpi=plot_dpi)
    plt.close()

def build_raster_index(static_tif_folder: str, filters: Tuple[str, ...]) -> Dict[str, str]:
    tif_paths = glob.glob(os.path.join(static_tif_folder, "*.tif"))
    if filters:
        tif_paths = [
            p for p in tif_paths
            if any(f.lower() in os.path.basename(p).lower() for f in filters)
        ]
    index: Dict[str, str] = {}
    for p in tif_paths:
        base = os.path.splitext(os.path.basename(p))[0]
        index[base] = p
    if not index:
        raise FileNotFoundError(f"No .tif files matched in {static_tif_folder} with filters={filters}")
    return index

def reconstruct_hybrid_grid_gat(
    model,
    grid_df: pd.DataFrame,
    fm: FeatureManager,
    grid_edges: torch.Tensor,
    ts: pd.Timestamp,
    train_norm: pd.DataFrame,
    y_mean: float,
    y_std: float,
    device,
    k_conn: Optional[int] = None,
    config: Config = None,
):
    """
    Given a trained model and a sampled/normalized grid_df for a timestamp ts,
    build the hybrid graph (sensors + grid), run a forward pass, and return
    predictions (in original temperature units) for the mesh nodes in the order
    of grid_df rows.
    """
    model.eval()
    if config is None:
        config = Config()
    if k_conn is None:
        k_conn = config.k_neighbors_sensors

    # Build unique training station table (should already be normalized in train_norm)
    cols = ["id_station", "latitude", "longitude"] + fm.all_raw_cols + ["lat_norm", "lon_norm"]
    unique_train_stations = (
        train_norm[cols].drop_duplicates("id_station").sort_values("id_station").reset_index(drop=True)
    )
    Ns = len(unique_train_stations)
    Ng = len(grid_df)

    # mapping id->idx in training node ordering
    id_to_idx = {sid: i for i, sid in enumerate(unique_train_stations["id_station"].tolist())}

    # Build sensor-sensor graph (same logic used during training)
    ei_ss, ew_ss = build_functional_sensor_graph(
        stations_df=unique_train_stations,
        k=(k_conn if k_conn is not None else config.k_neighbors_sensors),
        max_dist=(config.max_sensor_edge_km if config is not None else None),
        config=config,
        feature_cols=fm.all_raw_cols,
    )

    # Build sensor-grid edges (hybrid kNN)
    snap_df = train_norm[train_norm["dt"] == ts].merge(unique_train_stations[["id_station"]], on="id_station", how="inner")
    if snap_df.empty:
        # No sensor observations at this timestamp -> cold-start (temp_in zeros)
        snap_df = pd.DataFrame(columns=train_norm.columns)

    e_s2g_local, e_g2s_local = _knn_hybrid_sensor_to_grid_edges(
        snap_df=snap_df,
        grid_df=grid_df,
        fm=fm,
        cfg=config,
        k_eff=max(1, min(k_conn, max(1, Ns))),
        cols_for_ts=fm.get_cols_for_datetime(pd.Timestamp(ts), daily_fallback=config.daily_fallback),
    )

    # shift indices
    e_s2g = torch.stack([e_s2g_local[0], e_s2g_local[1] + Ns], dim=0) if e_s2g_local.numel() else torch.empty((2, 0), dtype=torch.long)
    e_g2s = torch.stack([e_g2s_local[0] + Ns, e_g2s_local[1]], dim=0) if e_g2s_local.numel() else torch.empty((2, 0), dtype=torch.long)

    # shift mesh edges
    e_grid_global = grid_edges.clone() + Ns if grid_edges.numel() else torch.empty((2, 0), dtype=torch.long)

    # combine edges
    parts = [p for p in [ei_ss, e_s2g, e_g2s, e_grid_global] if (p is not None and p.numel() > 0)]
    if not parts:
        raise RuntimeError("No edges available to construct the hybrid graph.")
    edge_index = torch.cat(parts, dim=1).long().to(device)

    # edge weights
    n_ss = ei_ss.shape[1] if ei_ss.numel() else 0
    rest_count = edge_index.shape[1] - n_ss
    ew_rest = torch.ones(rest_count, dtype=torch.float32).to(device)
    ew = torch.cat([ew_ss.to(device) if ew_ss is not None and ew_ss.numel() else torch.empty(0).to(device), ew_rest], dim=0).float()

    # Build node features:
    cols_for_ts = fm.get_cols_for_datetime(pd.Timestamp(ts), daily_fallback=config.daily_fallback)
    X_static_s = unique_train_stations[cols_for_ts].to_numpy(dtype=np.float32)
    X_static_g = grid_df[cols_for_ts].to_numpy(dtype=np.float32)

    t = pd.Timestamp(ts)
    tfeat = np.array([
        np.sin(2*np.pi*t.hour/24), np.cos(2*np.pi*t.hour/24),
        np.sin(4*np.pi*t.hour/24), np.cos(4*np.pi*t.hour/24),
        np.sin(2*np.pi*t.dayofyear/365), np.cos(2*np.pi*t.dayofyear/365)
    ], dtype=np.float32)
    X_time_s = np.tile(tfeat, (Ns, 1))
    X_time_g = np.tile(tfeat, (Ng, 1))

    X_coord_s = unique_train_stations[["lat_norm", "lon_norm"]].to_numpy(dtype=np.float32)
    X_coord_g = grid_df[["lat_norm", "lon_norm"]].to_numpy(dtype=np.float32)

    temp_in_s = np.zeros((Ns,), dtype=np.float32)
    snap_rows = train_norm[train_norm["dt"] == ts]
    if not snap_rows.empty:
        for _, r in snap_rows.iterrows():
            sid = r["id_station"]
            if sid in id_to_idx:
                i = id_to_idx[sid]
                temp_in_s[i] = (r["temperature"] - y_mean) / max(y_std, 1e-9)

    temp_in_g = np.zeros((Ng,), dtype=np.float32)

    X_s = np.concatenate([X_static_s, X_time_s, X_coord_s, temp_in_s.reshape(-1, 1)], axis=1) if Ns > 0 else np.zeros((0, fm.get_input_dim() + 6 + 2 + 1), dtype=np.float32)
    X_g = np.concatenate([X_static_g, X_time_g, X_coord_g, temp_in_g.reshape(-1, 1)], axis=1) if Ng > 0 else np.zeros((0, fm.get_input_dim() + 6 + 2 + 1), dtype=np.float32)
    X_all = np.concatenate([X_s, X_g], axis=0)
    x_tensor = torch.from_numpy(X_all).float().to(device)

    with torch.no_grad():
        n_static = fm.get_input_dim()
        n_time = 6
        x_static_time = x_tensor[:, : n_static + n_time]
        x_temp_in = x_tensor[:, -1:].to(x_static_time.dtype)
        x_sem = torch.cat([x_static_time, x_temp_in], dim=1)
        x_geo = x_tensor[:, n_static + n_time : n_static + n_time + 2]
        out = model(x_sem, x_geo, edge_index).detach().cpu().numpy().flatten()

    preds_grid_std = out[Ns:]
    preds_grid = preds_grid_std * y_std + y_mean
    return preds_grid

def ensemble_reconstruct_grid_at_ts(models_info, cfg, template_tif, ts, raster_index, k_conn=None, weighted=False):
    if k_conn is None:
        k_conn = cfg.k_neighbors_sensors

    # Build irregular mesh (cached)
    grid, grid_meta = build_irregular_mesh(cfg, template_tif, num_nodes=cfg.mesh_num_nodes)

    preds, weights = [], []
    for mi in models_info:
        grid_i = sample_static_tifs_for_grid(
            grid, cfg, mi["fm"], mi["static_stats"], mi["coord_scaler"], ts=pd.Timestamp(ts),
            raster_index=raster_index
        )
        grid_edges = build_mesh_edges_delaunay(grid_meta)
        p = reconstruct_hybrid_grid_gat(
            mi["model"], grid_i, mi["fm"], grid_edges, pd.Timestamp(ts),
            mi["train_norm"], mi["y_mean"], mi["y_std"], mi["device"], k_conn=k_conn, config=cfg
        )
        preds.append(p)
        if weighted:
            rmse = mi.get("rmse", np.nan)
            weights.append(1.0 / (float(rmse) + 1e-6) if np.isfinite(rmse) else 1.0)

    P = np.vstack(preds)
    if weighted and len(weights) == len(preds):
        w = np.array(weights, dtype=float)
        w = w / (w.sum() + 1e-12)
        ensemble_mean = (w[:, None] * P).sum(axis=0)
    else:
        ensemble_mean = np.nanmean(P, axis=0)
    ensemble_std = np.nanstd(P, axis=0)
    return grid, grid_meta, ensemble_mean, ensemble_std


def plot_irregular_mesh_result(grid_df, grid_meta, values, cfg, ts, template_tif, title_suffix="", cmap="magma", show_uncertainty=False, uncertainty=None):
    """
    Interpolates irregular mesh results onto a regular grid for visualization.
    """
    # Create target regular grid from template meta
    H, W = grid_meta["height"], grid_meta["width"]
    transform = grid_meta["transform"]
    
    # Generate coordinate grid for output image
    cols, rows = np.meshgrid(np.arange(W), np.arange(H))
    xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')
    xs = np.array(xs); ys = np.array(ys)
    
    # Reproject regular grid coords to Lat/Lon for interpolation compatibility
    grid_lons, grid_lats = rt_transform(grid_meta["crs"].to_string(), cfg.sensor_crs, xs.flatten(), ys.flatten())
    grid_lons = np.array(grid_lons).reshape(H, W)
    grid_lats = np.array(grid_lats).reshape(H, W)
    
    # Interpolate
    points = np.column_stack((grid_df["longitude"].to_numpy(), grid_df["latitude"].to_numpy()))
    
    img = griddata(points, values, (grid_lons, grid_lats), method='linear')
    if np.isnan(img).any():
        # Fill edges with nearest to avoid white borders
        mask = np.isnan(img)
        img[mask] = griddata(points, values, (grid_lons[mask], grid_lats[mask]), method='nearest')

    img_s = smooth_image(img, cfg.smooth_sigma_px, cfg.smooth_passes)
    
    plt.figure(figsize=(7,6))
    plt.imshow(img_s, vmin=cfg.temp_min, vmax=cfg.temp_max, cmap=cmap)
    plt.colorbar(label="Predicted Ta (°C)")
    plt.title(f"Irregular Mesh Recon | {pd.Timestamp(ts)}{title_suffix}", fontweight='bold'); plt.axis("off")
    out_base = os.path.join(cfg.output_dir, f"mesh_recon_{pd.Timestamp(ts).strftime('%Y%m%d_%H%M%S')}")
    plt.savefig(out_base + ".png", dpi=cfg.plot_dpi, bbox_inches='tight'); plt.close()
    
    if cfg.export_recon_tifs:
        with rasterio.open(template_tif) as src: meta = src.meta.copy()
        meta.update({"driver": "GTiff", "count": 1, "dtype": "float32", "compress": "deflate"})
        with rasterio.open(out_base + ".tif", "w", **meta) as dst:
            dst.write(img_s.astype(np.float32), 1)

    if show_uncertainty and uncertainty is not None:
        u_img = griddata(points, uncertainty, (grid_lons, grid_lats), method='linear')
        if np.isnan(u_img).any():
             mask = np.isnan(u_img)
             u_img[mask] = griddata(points, uncertainty, (grid_lons[mask], grid_lats[mask]), method='nearest')
        u_img_s = smooth_image(u_img, cfg.smooth_sigma_px, cfg.smooth_passes)
        
        plt.figure(figsize=(7,6)); plt.imshow(u_img_s, cmap="viridis")
        plt.colorbar(label="Ensemble Std (°C)")
        plt.title(f"Mesh Uncertainty | {pd.Timestamp(ts)}", fontweight='bold'); plt.axis("off")
        plt.savefig(out_base + "_uncertainty.png", dpi=cfg.plot_dpi, bbox_inches='tight'); plt.close()

# --- SPATIAL RANGE FUNCTION ---
def timesteps_with_spatial_range(df: pd.DataFrame, min_range: float = 4.0, max_range: float = 6.0) -> List[pd.Timestamp]:
    agg = df.groupby("dt")["temperature"].agg(["min", "max"])
    ranges = agg["max"] - agg["min"]
    eligible = ranges[(ranges >= min_range) & (ranges <= max_range)].index
    return list(pd.to_datetime(eligible))

###########################################
if __name__ == "__main__":
    cfg = Config()
    set_seed(cfg.seed)

    print("Loading sensor data (native timestamp resolution)...")
    big_df = load_all_csvs(cfg)
    print(f"Loaded {len(big_df)} rows across {big_df['id_station'].nunique()} stations and {big_df['dt'].nunique()} timestamps.")
    print(f"Date range: {big_df['dt'].min()} to {big_df['dt'].max()}")

    print("Sampling static/daily/monthly TIFs for stations...")
    final_df, fm = sample_static_and_init_features(big_df, cfg)
    print(f"Final dataframe rows: {len(final_df)}; static/daily/monthly bases: {fm.get_input_dim()}")

    raster_index = build_raster_index(cfg.static_tif_folder, cfg.static_name_filters)

    all_loss_histories: List[List[float]] = []
    models_info: List[Dict[str, Any]] = []

    stations = final_df["id_station"].unique().tolist()
    random.shuffle(stations)

    # --- LOOCV LOOP ---
    for hold_id in stations:
        cfg.output_dir = os.path.join(cfg.base_output_dir, str(hold_id).replace(":", "_"))
        os.makedirs(cfg.output_dir, exist_ok=True)
        print(f"\n=== Fold for holdout: {hold_id} ===")

        train_df = final_df[final_df["id_station"] != hold_id].copy()
        hold_df  = final_df[final_df["id_station"] == hold_id].copy()

        static_stats, coord_scaler = fit_normalizers_fast(train_df, fm.all_raw_cols)
        train_norm = normalize_data(train_df, fm.all_raw_cols, static_stats, coord_scaler)
        hold_norm  = normalize_data(hold_df, fm.all_raw_cols, static_stats, coord_scaler)

        cols = ["id_station","latitude","longitude"] + fm.all_raw_cols + ["lat_norm","lon_norm"]
        unique_train_stations = train_norm[cols].drop_duplicates("id_station").sort_values("id_station").reset_index(drop=True)
        id_to_idx = {sid: i for i, sid in enumerate(unique_train_stations["id_station"])}
        
        ei, ew = build_functional_sensor_graph(unique_train_stations, cfg.k_neighbors_sensors, cfg.max_sensor_edge_km, cfg, fm.all_raw_cols)
        print(f"Train stations: {len(unique_train_stations)}, edges: {ei.shape[1]}")

        model, history, y_mean, y_std, device = train_model(train_norm, unique_train_stations, id_to_idx, fm, ei, ew, cfg)
        print(f"Training complete. Last loss: {history['train_loss'][-1]:.4f}")
        all_loss_histories.append(history["train_loss"])
        plt.figure(figsize=(8,4)); plt.plot(history["train_loss"]); plt.tight_layout()
        plt.savefig(os.path.join(cfg.output_dir, "training_loss.png"), dpi=cfg.plot_dpi); plt.close()
        save_trained_model(cfg.output_dir, model, y_mean, y_std, fm, cfg)

        rmse, mae = evaluate_holdout(model, train_norm, hold_norm, unique_train_stations, id_to_idx, fm, ei, ew, y_mean, y_std, cfg, device, str(hold_id))
        print(f"Holdout evaluation -> RMSE={rmse}, MAE={mae}")

        models_info.append({
            "model": model.to(device),
            "device": device,
            "y_mean": y_mean,
            "y_std": y_std,
            "fm": fm,
            "static_stats": static_stats,
            "coord_scaler": coord_scaler,
            "train_norm": train_norm,
            "rmse": rmse,
            "hold_id": str(hold_id),
        })

    plot_mean_loss_across_folds(all_loss_histories, cfg.base_output_dir, plot_dpi=cfg.plot_dpi)

    # --- ENSEMBLE RECONSTRUCTION ---
    tifs = filter_tifs_by_name(glob.glob(os.path.join(cfg.static_tif_folder, "*.tif")), cfg.static_name_filters)
    template_tif = cfg.template_tif_for_grid or (tifs[0] if tifs else None)
    assert template_tif and os.path.exists(template_tif), "Provide a valid template TIF."

    times_by_model = [set(mi["train_norm"]["dt"].dropna().unique()) for mi in models_info]
    common_times = set.intersection(*times_by_model) if times_by_model else set()

    eligible_ts = set(timesteps_with_spatial_range(final_df, min_range=4.0, max_range=6.0))
    candidate_times = sorted(list(common_times & eligible_ts))

    if len(candidate_times) == 0:
        print("No common times with 4–6°C spatial range found; falling back to common_times.")
        candidate_times = sorted(list(common_times))
    if len(candidate_times) == 0 and times_by_model:
        union_times = sorted(list(set.union(*times_by_model)))
        candidate_times = union_times

    picks = random.sample(candidate_times, k=min(cfg.n_export_samples, len(candidate_times)))
    cfg.output_dir = os.path.join(cfg.base_output_dir, "ensemble")
    os.makedirs(cfg.output_dir, exist_ok=True)

    print(f"Exporting {len(picks)} samples using Irregular Mesh...")
    for ts in picks:
        grid_df, grid_meta, mean_pred, std_pred = ensemble_reconstruct_grid_at_ts(
            models_info, cfg, template_tif, ts, raster_index=raster_index, weighted=False
        )
        # Interpolate irregular mesh back to regular image for plotting
        plot_irregular_mesh_result(
            grid_df, grid_meta, mean_pred, cfg, ts, template_tif, 
            title_suffix=" (Mean)", show_uncertainty=True, uncertainty=std_pred
        )

    print("Ensemble irregular mesh maps exported.")

Loading sensor data (native timestamp resolution)...
Loaded 10800 rows across 15 stations and 720 timestamps.
Date range: 2023-08-01 00:00:00 to 2023-08-30 23:00:00
Sampling static/daily/monthly TIFs for stations...
Final dataframe rows: 10800; static/daily/monthly bases: 5

=== Fold for holdout: S-THC 21317948:21326445 ===
Train stations: 14, edges: 168
Training complete. Last loss: 0.0330
Saved model to: C:\Downloads\UHI_research\Output_LOOCV_Irregular_MeshTa\S-THC 21317948_21326445\model_state.pth
Holdout S-THC 21317948:21326445: RMSE=0.550, MAE=0.360
Saved holdout observed vs predicted CSV to: C:\Downloads\UHI_research\Output_LOOCV_Irregular_MeshTa\S-THC 21317948_21326445\holdout_observed_vs_predicted.csv
Holdout evaluation -> RMSE=0.5501812796716947, MAE=0.3600922203270101

=== Fold for holdout: S-THC 21317959:21326443 ===
Train stations: 14, edges: 168
Training complete. Last loss: 0.0363
Saved model to: C:\Downloads\UHI_research\Output_LOOCV_Irregular_MeshTa\S-THC 21317959_21326

In [4]:
# === Replace existing candidate_times / picks logic with this block ===
highrange_csv = r"C:\Downloads\UHI_research\original_2023\filtered_2023_aug_hourly_highrange.csv"
if not os.path.exists(highrange_csv):
    raise FileNotFoundError(f"High-range CSV not found: {highrange_csv}")

# Load high-range timestamps (support common column names)
_hr = pd.read_csv(highrange_csv)
if "dt" in _hr.columns:
    _hr["dt"] = pd.to_datetime(_hr["dt"], errors="coerce")
elif "date" in _hr.columns:
    _hr["dt"] = pd.to_datetime(_hr["date"], errors="coerce")
else:
    # fallback: parse first column as dates
    first_col = _hr.columns[0]
    _hr["dt"] = pd.to_datetime(_hr[first_col], errors="coerce")

highrange_ts = sorted(pd.to_datetime(_hr["dt"].dropna().unique()))

# Compute union of times seen by models (so we avoid times none of the models saw)
times_by_model = [set(mi["train_norm"]["dt"].dropna().unique()) for mi in models_info]
union_times = set.union(*times_by_model) if times_by_model else set()

# Intersect with highrange list so we only reconstruct for requested timesteps that models can handle
selected_ts = [ts for ts in highrange_ts if ts in union_times]

if not selected_ts:
    # If intersection empty, warn and fall back to the highrange timestamps themselves.
    # Note: models may perform cold-start predictions for times they didn't train on,
    # but FeatureManager.get_cols_for_datetime may raise if rasters are missing for that date.
    print("Warning: none of the high-range timestamps were found in models' training times.")
    print("Falling back to all high-range timestamps (cold-start predictions possible).")
    selected_ts = highrange_ts

print(f"Reconstructing for {len(selected_ts)} high-range timestamps (from {highrange_csv}).")

# Use deterministic ordering (or random.sample if you want a subset)
selected_ts = sorted(selected_ts)

# Run ensemble reconstruction for each selected timestamp
cfg.output_dir = os.path.join(cfg.base_output_dir, "ensemble")
os.makedirs(cfg.output_dir, exist_ok=True)

for ts in selected_ts:
    try:
        grid_df, grid_meta, mean_pred, std_pred = ensemble_reconstruct_grid_at_ts(
            models_info, cfg, template_tif, ts, raster_index=raster_index, weighted=False
        )
        plot_irregular_mesh_result(
            grid_df, grid_meta, mean_pred, cfg, ts, template_tif,
            title_suffix=" (Mean)", show_uncertainty=True, uncertainty=std_pred
        )
        print(f"Exported ensemble reconstruction for {ts}")
    except Exception as e:
        print(f"Failed to reconstruct for {ts}: {e}")

Reconstructing for 73 high-range timestamps (from C:\Downloads\UHI_research\original_2023\filtered_2023_aug_hourly_highrange.csv).
Exported ensemble reconstruction for 2023-08-01 06:00:00
Exported ensemble reconstruction for 2023-08-01 07:00:00
Exported ensemble reconstruction for 2023-08-01 08:00:00
Exported ensemble reconstruction for 2023-08-01 21:00:00
Exported ensemble reconstruction for 2023-08-02 06:00:00
Exported ensemble reconstruction for 2023-08-02 07:00:00
Exported ensemble reconstruction for 2023-08-02 08:00:00
Exported ensemble reconstruction for 2023-08-02 13:00:00
Exported ensemble reconstruction for 2023-08-02 14:00:00
Exported ensemble reconstruction for 2023-08-02 15:00:00
Exported ensemble reconstruction for 2023-08-02 17:00:00
Exported ensemble reconstruction for 2023-08-02 19:00:00
Exported ensemble reconstruction for 2023-08-02 22:00:00
Exported ensemble reconstruction for 2023-08-04 16:00:00
Exported ensemble reconstruction for 2023-08-06 07:00:00
Exported ensem